In [47]:
import pandas as pd
import awswrangler as wr
import boto3
import os

os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
os.environ["AWS_PROFILE"] = "sandbox"

# Config values from your setup
glue_database_name = "nyc_taxi"
glue_database_s3_bucket_name = "nyc-taxi-datalake-glue-nyc-taxi"
table_name = "customers"
table_s3_path = f"s3://{glue_database_s3_bucket_name}/{glue_database_name}/{table_name}/"
temp_s3_path = f"s3://{glue_database_s3_bucket_name}/temp/{glue_database_name}/{table_name}/"

# Create Glue database if it doesn't exist
wr.catalog.create_database(name=glue_database_name, exist_ok=True)

# Step 1: CREATE TABLE statement for Iceberg table
ddl = f"""
CREATE TABLE IF NOT EXISTS {glue_database_name}.{table_name} (
    customer_id BIGINT,
    name STRING,
    email STRING,
    status STRING
)
LOCATION '{table_s3_path}'
TBLPROPERTIES (
  'table_type'='ICEBERG', 
  'format'='parquet'
)
"""

# wr.athena.start_query_execution(sql=f'DELETE FROM {glue_database_name}.{table_name} WHERE TRUE', database=glue_database_name)
# wr.athena.start_query_execution(sql=f'DROP TABLE IF EXISTS {glue_database_name}.customers', database=glue_database_name)
wr.athena.start_query_execution(sql=ddl, database=glue_database_name)
print("✅ Table created or already exists.")

df1 = pd.DataFrame([
    {"customer_id": 1, "name": "Alice", "email": "alice@example.com", "status": "active"},
    {"customer_id": 2, "name": "Bob", "email": "bob@example.com", "status": "active"},
])

df2 = pd.DataFrame([
    {"customer_id": 2, "name": "Bob", "email": "bobby@example.com", "status": "inactive"},
    {"customer_id": 3, "name": "Charlie", "email": "charlie@example.com", "status": "active"},
])

df3 = pd.DataFrame([
    {"customer_id": 1, "name": "Alice Smith", "email": "alice.smith@example.com", "status": "active"},
    {"customer_id": 4, "name": "Dana", "email": "dana@example.com", "status": "active"},
])

# Step 3: Write to Athena Iceberg table with upsert-style merging
for i, df in enumerate([df1, df2, df3], 1):
    print(f"🔄 Iteration {i}: Merging DataFrame:\n{df}\n")
    wr.athena.to_iceberg(
        df=df,
        database=glue_database_name,
        table=table_name,
        # files are uploaded to this path; then a temp plain Glue-Parquet table is created over this; then MERGE temp -> table is called.
        temp_path=temp_s3_path,
        mode="append", # "overwrite" will drop the whole table, even if you specify merge_cols... so append is what we want for idempotent, incremental upserts
        merge_condition="update",
        # schema_evolution=True,
        merge_cols=["customer_id"]
    )
    
    # Query the table to see current state
    query_result = wr.athena.read_sql_query(
        sql=f"SELECT * FROM {glue_database_name}.{table_name} ORDER BY customer_id",
        database=glue_database_name
    )
    print(f"📊 Table state after iteration {i}:")
    print(query_result)
    
    # if we do not delete the parquet files in the temp path, then they pile up
    # after each .to_iceberg() call. When the MERGE query is run by athena, ALL
    # the files ever created here in temp will be considered in the MERGE...
    # which means, you are going to have duplicates and the merge will fail
    # it's also just ridiculously confusing. Honestly, .to_iceberg() doesn't make
    # sense to call in a loop. Just concatenate all the DFs and do it all at once.
    # Otherwise, you should really just write all the parquet files to S3 yourself
    # in a "temp path" and then write a MERGE query yourself to append them into iceberg
    # all at once. FYI if you do this, apparently the order of the column names in the merge
    # statement has to be EXACTLY the same order as the columns are defined in the Glue/Iceberg table.
    # So, watch out for that. The .to_iceberg() query does this for you by querying the target table
    # to get the order of the columns first.
    wr.s3.delete_objects(path=temp_s3_path)
    print(f"🧹 Cleaned up temp files in {temp_s3_path}")
    print("-" * 50)

print("✅ Merge complete.")
print(f"🔍 Query in Athena:\nSELECT * FROM {glue_database_name}.{table_name};")

✅ Table created or already exists.
🔄 Iteration 1: Merging DataFrame:
   customer_id   name              email  status
0            1  Alice  alice@example.com  active
1            2    Bob    bob@example.com  active

📊 Table state after iteration 1:
   customer_id     name                email  status
0            1    Alice    alice@example.com  active
1            2      Bob      bob@example.com  active
2            3  Charlie  charlie@example.com  active
3            4     Dana     dana@example.com  active
📊 Table state after iteration 1:
   customer_id     name                email  status
0            1    Alice    alice@example.com  active
1            2      Bob      bob@example.com  active
2            3  Charlie  charlie@example.com  active
3            4     Dana     dana@example.com  active
🧹 Cleaned up temp files in s3://nyc-taxi-datalake-glue-nyc-taxi/temp/nyc_taxi/customers/
--------------------------------------------------
🔄 Iteration 2: Merging DataFrame:
   customer_i

In [34]:
wr.athena.read_sql_query(
    sql=f"""
        SELECT * 
        FROM {glue_database_name}.{table_name} 
        FOR TIMESTAMP AS OF (current_timestamp - interval '5' minute)
        ORDER BY customer_id
    """,
    database=glue_database_name
)

,customer_id,name,email,status
0,1,Alice,alice@example.com,active
1,1,Alice,alice@example.com,active
2,1,Alice,alice@example.com,active
3,1,Alice,alice@example.com,active
4,1,Alice,alice@example.com,active
5,1,Alice,alice@example.com,active
6,1,Alice,alice@example.com,active
7,1,Alice,alice@example.com,active
8,1,Alice,alice@example.com,active
9,1,Alice,alice@example.com,active


In [39]:
# if you delete the objects in an S3 bucket with an iceberg / glue table, then 
# NO athena operations will work on that table anymore. No creates, selects, drops, deletes, etc.
# You have to formally delete the table using AWS API's to fully delete the remnants of it,
# and then you can re-create it if you want.

import boto3

client = boto3.client("glue")

client.delete_table(
    DatabaseName="nyc_taxi",
    Name="customers"
)

{'ResponseMetadata': {'RequestId': '71010c46-548e-4a10-b5fa-a57faa7ea569',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sat, 21 Jun 2025 06:30:18 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-amzn-requestid': '71010c46-548e-4a10-b5fa-a57faa7ea569',
   'cache-control': 'no-cache'},
  'RetryAttempts': 0}}